In [ ]:
dist_names = ['left', 'up', 'right', 'down']
dict_vect = {'left': [-1, 0], 'up': [0, -1], 'right': [1, 0], 'down': [0, 1]}
moves_dict = {'left': 0, 'up': 1, 'right': 2, 'down': 3}

def get_is_wall(allowed_moves, last_move):
    result = np.zeros((5,))
    for current_move in allowed_moves:
        result[moves_dict[current_move]] = 1
    result[4] = int(last_move in allowed_moves)
    result = 1 - result
    return result

In [ ]:
class Pacman(pygame.sprite.Sprite):
    # Сбрасываем reward при переинициализации
        self.reward = 0
        self.old_dist = -1
        self.old_dist_ghost = -1
        self.pacman_time = 0
        self.last_ate_time = 0

    def get_q_table(self):
        return self.Q

    # Check what moves are allowed from this position
    def get_allowed_moves(self):
        """
        Check if pacman is allowed to move here
        """
        global MAP
        self.allowed_moves = []

        # check walls
        if MAP[self.y][self.x-1] < 16:
            self.allowed_moves.append("left")
        if (self.x+1 < 28 and MAP[self.y][self.x+1] < 16) or (self.x == 27 and self.direction == "right"):
            self.allowed_moves.append("right")
        if MAP[self.y - 1][self.x] < 16:
            self.allowed_moves.append("up")
        if MAP[self.y + 1][self.x] < 16:
            self.allowed_moves.append("down")

    def check_pacgums(self):
        global MAP
        chase = False
        if MAP[self.y][self.x] == 1:
            self.game.score += 10
            MAP[self.y][self.x] = 0
            self.game.pacgums -= 1
            self.miss_loops = 5
        return chase

    def change_mode(self):
        """
        Two modes for pacman: chase or normal
        """
        self.mode_changed = False
        current_time = time.time()

        # Enter chase mode
        if self.check_pacgums():
            self.mode = "chase"
            self.start_time = current_time
            self.mode_changed = True
            for ghost in self.game.Ghosts:
                if ghost.mode != "eaten":
                    ghost.change_mode("runaway")

        # rotate between modes based on timer
        else:
            mode_time = PACMAN_TIMERS[self.mode]
            if current_time - self.start_time > mode_time:
                if self.mode == "chase":
                    # Reinit ghosts in a row counter
                    self.game.ghosts_in_a_row = 0
                    self.mode = "normal"
                self.start_time = current_time
                self.mode_changed = True


In [ ]:
def update(self):
        global MAP
        current_speed = self.speed
        if self.miss_loops:
            current_speed -= self.miss_loops
            self.miss_loops = 0

        # если следующий шаг превышает следующую клетку
        if self.direction == 'left' and self.real_x-current_speed < (self.x-1)*24*10:
            next_speed = self.real_x - (self.x-1)*240
        elif self.direction == 'right' and self.real_x+current_speed > (self.x+1)*24*10:
            next_speed = (self.x+1)*240  - self.real_x
        elif self.direction == 'up' and self.real_y-current_speed < (self.y-1)*24*10:
            next_speed = self.real_y - (self.y-1)*240
        elif self.direction == 'down' and self.real_y+current_speed > (self.y+1)*24*10:
            next_speed = (self.y+1)*240 - self.real_y
        else:
            next_speed = current_speed

        next_loops = (next_speed, current_speed-next_speed)
        for speed in next_loops:
            if speed == 0:
                continue
                    
            # Выбираем направление только когда мы на координатах карты
            if (self.real_x / 10) % 24 == 0 and (self.real_y / 10) % 24 == 0:
                self.pacman_time += 1
                self.reward = 0
                self.x = int(self.real_x / 10 / 24)
                self.y = int(self.real_y / 10 / 24)
                self.change_mode()
                self.get_allowed_moves()
                
                #СОХРАНЯЕМ СТАРОЕ СОСТОЯНИЕ И ДЕЙСТВИЕ 
                old_state = self.current_state
                old_action = self.action
                
                if self.game.score > self.game.old_score:
                    self.last_ate_time = self.pacman_time
                    self.reward += self.game.score - self.game.old_score

                # Находим ближайший пак-гам (внутренний метод)
                y_length = len(MAP)
                x_length = len(MAP[0])
                pacgum_list = []
                for y in range(y_length):
                    for x in range(x_length):
                        if MAP[y][x] == 1:
                            dist = ((x - self.x) ** 2 + (y - self.y) ** 2) ** 0.5
                            pacgum_list.append({'x': x, 'y': y, 'd': dist})
                
                if len(pacgum_list) > 0:
                    ind = np.argmin([pacgum_list[i]['d'] for i in range(len(pacgum_list))])
                    current_dist = pacgum_list[ind]['d']
                    pg_x = pacgum_list[ind]['x']
                    pg_y = pacgum_list[ind]['y']

                    # Награда за приближение к точке
                    if current_dist < self.old_dist and self.pacman_time - self.last_ate_time < 50:
                        self.reward += self.old_dist - current_dist
                    if current_dist > self.old_dist and self.pacman_time - self.last_ate_time > 50:
                        self.reward -= 0.5
                    
                    self.old_dist = pacgum_list[ind]['d']
                    
                    # Находим ближайшего призрака (внутренний метод)
                    ghost_list = []
                    for ghost in self.game.Ghosts:
                        dist = ((ghost.x - self.x) ** 2 + (ghost.y - self.y) ** 2) ** 0.5
                        ghost_list.append({'x': ghost.x, 'y': ghost.y, 'd': dist})
                    
                    if len(ghost_list) > 0:
                        ind = np.argmin([ghost_list[i]['d'] for i in range(len(ghost_list))])
                        ghost_x = ghost_list[ind]['x']
                        ghost_y = ghost_list[ind]['y']
                        
                        current_dist_ghost = ghost_list[ind]['d']
                        # Штраф за приближение призрака
                        if current_dist_ghost < self.old_dist_ghost:
                            self.reward -= 3 * (self.old_dist_ghost - current_dist_ghost)
                        
                        self.old_dist_ghost = ghost_list[ind]['d']
                        
                        #  ВЫЧИСЛЯЕМ ТЕКУЩЕЕ СОСТОЯНИЕ
                        self.current_state = (xy_state(self.x, self.y, ghost_x, ghost_y) * 8 + 
                                            xy_state(self.x, self.y, pg_x, pg_y)) * 3 + len(self.allowed_moves) - 2
                        
                        # ВЫБОР ДЕЙСТВИЯ (ε-greedy стратегия) 
                        self.is_wall = get_is_wall(self.allowed_moves, self.last_move)
                        
                        # Получаем список допустимых действий
                        valid_actions = []
                        for i in range(5):  # 5 действий: left(0), up(1), right(2), down(3), stay(4)
                            if self.is_wall[i] == 0:  # 0 = можно двигаться
                                valid_actions.append(i)
                        
                        # ε-greedy выбор
                        if random.random() < self.epsilon and valid_actions:
                            # Случайное исследование
                            self.action = random.choice(valid_actions)
                        else:
                            # Эксплуатация: выбираем лучшее действие
                            q_values = self.Q[self.current_state, :].copy()
                            
                            # Устанавливаем очень низкие значения для запрещенных действий
                            q_values[self.is_wall == 1] = -1e6
                            
                            # Находим максимальное Q-значение
                            max_q = np.max(q_values)
                            
                            # Если несколько действий имеют одинаковое максимальное значение
                            best_actions = np.where(q_values == max_q)[0]
                            
                            # Выбираем случайное из лучших действий
                            if len(best_actions) > 0:
                                self.action = random.choice(best_actions.tolist())
                            else:
                                # Если все действия запрещены (не должно быть), выбираем stay
                                self.action = 4
                        
                        # ОБНОВЛЯЕМ НАПРАВЛЕНИЕ ДВИЖЕНИЯ 
                        if self.action < 4:
                            self.direction = dist_names[self.action]
                        else:
                            self.direction = self.last_move
                        
                        #  ОБНОВЛЕНИЕ Q-ТАБЛИЦЫ (Q-learning) 
                        # Только если у нас есть предыдущее состояние и действие
                        if old_state is not None and old_action is not None:
                            # Получаем текущее Q-значение
                            current_q = self.Q[old_state, old_action]
                            
                            # Находим максимальное Q-значение для текущего (нового) состояния
                            next_q_values = self.Q[self.current_state, :].copy()
                            
                            # Учитываем стены для следующего состояния
                            next_is_wall = get_is_wall(self.allowed_moves, self.last_move)
                            next_q_values[next_is_wall == 1] = -1e6
                            
                            # Находим максимальное допустимое Q-значение
                            max_next_q = np.max(next_q_values) if len(next_q_values) > 0 else 0
                            
                            # Формула Q-learning: Q(s,a) = Q(s,a) + α * (reward + γ * maxQ(s',a') - Q(s,a))
                            new_q = current_q + self.alpha * (self.reward + self.gamma * max_next_q - current_q)
                            self.Q[old_state, old_action] = new_q
                            
                            # Уменьшаем epsilon (исследование)
                            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
                        
                        self.game.old_score = self.game.score
                
                # Управление с клавиатуры (для отладки)
                keys = pygame.key.get_pressed()
                if keys[pygame.K_LEFT]:
                    if MAP[self.y][self.x-1] < 16:
                        self.direction = "left"
                elif keys[pygame.K_RIGHT]:
                    if self.x+1 < 28 and MAP[self.y][self.x+1] < 16:
                        self.direction = "right"
                elif keys[pygame.K_UP]:
                    if MAP[self.y - 1][self.x] < 16:
                        self.direction = "up"
                elif keys[pygame.K_DOWN]:
                    if self.y + 1 < 30 and MAP[self.y + 1][self.x] < 16:
                        self.direction = "down"

            # 6. ВЫПОЛНЕНИЕ ВЫБРАННОГО ДЕЙСТВИЯ (ПЕРЕМЕЩЕНИЕ)
            self.last_moved = True
            if self.direction == "left" and "left" in self.allowed_moves:
                self.last_move = 'left'
                self.real_x -= speed
                self.rect.x = round(self.real_x/10)
                # переход в правый туннель
                if self.rect.x <= -24:
                    self.rect.x = self.game.WIDTH-24
                    self.real_x = self.rect.x*10
            elif self.direction == "right" and "right" in self.allowed_moves:
                self.last_move = 'right'
                self.real_x += speed
                self.rect.x = round(self.real_x/10)
                # переход в левый туннель
                if self.rect.x >= self.game.WIDTH:
                    self.rect.x = -24
                    self.real_x = -240
            elif self.direction == "up" and "up" in self.allowed_moves:
                self.last_move = 'up'
                self.real_y -= speed
                self.rect.y = round(self.real_y/10)
            elif self.direction == "down" and "down" in self.allowed_moves:
                self.last_move = 'down'
                self.real_y += speed
                self.rect.y = round(self.real_y/10)
            else:
                self.last_moved = False
                self.last_move = ''
    
    # Обновляем позицию спрайта
        if self.last_moved:
            self.count_moves += 1
            self.rect.x = round(self.real_x/10) - 4
            self.rect.y = round(self.real_y/10) - 4
        
        # Обновляем анимацию
        if self.direction:
            if self.last_moved:
                self.image = self.game.Pacman_pics[self.direction][self.count_moves % 3 + 1]
            else:
                self.image = self.game.Pacman_pics[self.direction][2]

In [ ]:
class Game:
    pygame.init()
        self.parse_args()
        display_infos = pygame.display.Info()
        y_resolution = display_infos.current_h
        if y_resolution < 800:
            self.scale = 0.75
        self.screen = pygame.display.set_mode((int(self.FULL_WIDTH * self.scale), int(self.FULL_HEIGHT * self.scale)))
        self.fake_screen = pygame.Surface((self.FULL_WIDTH, self.FULL_HEIGHT))
        self.surface = pygame.Surface((self.WIDTH, self.HEIGHT))
        self.top = pygame.Surface((self.WIDTH, 32))
        self.bottom = pygame.Surface((self.WIDTH, 40))
        self.load_bitmaps()
        self.all_sprites = pygame.sprite.Group()
        self.all_ghosts = pygame.sprite.Group()
        self.bonus = None
        self.red = Ghost(self, 13, 11, 'red', "scatter")
        for color in ("blue", "yellow", "pink"):
            setattr(self, color, Ghost(self, JAIL[color][0], JAIL[color][1], color, "jail") )
        self.Ghosts = []
        self.Ghosts.append(self.pink)
        # self.Ghosts.append(self.red)
        # self.Ghosts.append(self.yellow)
        # self.Ghosts.append(self.blue)
        self.pacman = Pacman(self, PACMAN_POS[0], PACMAN_POS[1], input_Q)
        self.all_sprites.add(self.pacman)
        self.all_sprites.add(self.Ghosts)
        self.all_ghosts.add(self.Ghosts)

    def parse_args(self):
        parser = argparse.ArgumentParser()
        parser.add_argument('-t', '--theme', help='Theme name', default='default')
        args, unknown = parser.parse_known_args()
        # args = parser.parse_args()
        if not os.path.isdir(os.path.join(os.path.dirname(os.path.abspath('')), 'themes/'+args.theme)):
            self.theme = 'default'
        else:
            self.theme = args.theme

    def load_bitmaps(self):
        global MAP
        game_folder = os.getcwd()
        img_folder = os.path.join(game_folder, 'themes\\'+self.theme+'\\img')
        self.Frightened_ghost = dict()
        self.Frightened_ghost[1] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_1.png')).convert(),(32,32))
        self.Frightened_ghost[2] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_2.png')).convert(),(32,32))
        self.Frightened_ghost[1].set_colorkey(BLACK)
        self.Frightened_ghost[2].set_colorkey(BLACK)
        self.Frightened_ghost_blinking = dict()
        self.Frightened_ghost_blinking[1] = self.Frightened_ghost[1]
        self.Frightened_ghost_blinking[2] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_4.png')).convert(),(32,32))
        self.Frightened_ghost_blinking[3] = self.Frightened_ghost[2]
        self.Frightened_ghost_blinking[4] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_3.png')).convert(),(32,32))
        self.Frightened_ghost_blinking[2].set_colorkey(BLACK)
        self.Frightened_ghost_blinking[4].set_colorkey(BLACK)
        self.Ghost_pics = dict()
        for color in ('red', 'yellow', 'blue', 'pink'):
            self.Ghost_pics[color] = dict()
            for direction in ('left', 'right', 'up', 'down'):
                self.Ghost_pics[color][direction] = dict()
                for i in range(1, 3):
                    self.Ghost_pics[color][direction][i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, color+'_'+direction+'_ghost_'+str(i)+'.png')).convert(),(32,32))
                    self.Ghost_pics[color][direction][i].set_colorkey(BLACK)
        self.Ghost_eyes = dict()
        for direction in ('left', 'right', 'up', 'down'):
            self.Ghost_eyes[direction] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'ghost_eyes_'+direction+'.png')).convert(),(32,32))
            self.Ghost_eyes[direction].set_colorkey(BLACK)
        self.Pacman_pics = dict()
        for direction in ('left', 'right', 'up', 'down'):
            self.Pacman_pics[direction] = dict()
            for i in range(1, 4):
                self.Pacman_pics[direction][i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'pacman_'+direction+'_'+str(i)+'.png')).convert(),(32,32))
                self.Pacman_pics[direction][i].set_colorkey(BLACK)
        self.Dead_pacman = dict()
        for i in range(1, 11):
            self.Dead_pacman[i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'pacman_dead_'+str(i)+'.png')).convert(),(32,32))
            self.Dead_pacman[i].set_colorkey(BLACK)
        self.Scores = dict()
        for score in (100, 200, 300, 400, 500, 700, 800, 1000, 1600, 2000, 3000, 5000):
            self.Scores[score] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, str(score)+'.png')).convert(),(32,32))
            self.Scores[score].set_colorkey(BLACK)
        self.Walls = dict()
        for l in MAP:
            for c in l:
                if c not in self.Walls:
                    png = os.path.join(img_folder, str(c) + ".png")
                    if os.path.exists(png):
                        self.Walls[c] = pygame.image.load(png).convert()
        for bonus in range(7,15):
            self.Walls[bonus] = pygame.image.load(os.path.join(img_folder, str(bonus)+'.png')).convert()
            self.Walls[bonus].set_colorkey(BLACK)
        self.Ready = pygame.image.load(os.path.join(img_folder, 'ready.png')).convert()
        self.Ready.set_colorkey(BLACK)

    def display_board_game(self):
        self.clear_all_surfaces()
        self.display_score()
        self.display_map(self.surface)
        self.all_sprites.draw(self.surface)
        self.display_lifes(self.bottom)
        self.blit_all_surfaces()

    def display_text(self, my_surface, my_text, pos_x, pos_y, color=WHITE):
        font = pygame.font.Font('RetroGaming.ttf', 18)
        text = font.render(my_text, True, color)
        my_surface.blit(text, (pos_x, pos_y))

    def display_score(self):
        self.display_text(self.top, "Player 1     Score: {}".format(self.score), 24, 6)

    def display_lifes(self, my_surface):
        for life in range(0, self.lifes - 1):
            my_surface.blit(self.Pacman_pics['right'][2], (life*32+24, 4))

    def start_game(self):
        i = 0
        while i<4:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    break
            i += 1 
            self.clear_all_surfaces()
            self.display_map(self.surface)
            self.display_score()
            self.display_map(self.surface)
            self.all_sprites.draw(self.surface)
            self.display_lifes(self.bottom)
            self.surface.blit(self.Ready, (11*24, 17*24))
            self.blit_all_surfaces()
            pygame.display.flip()
            time.sleep(0.001)

    def loose_life(self):
        global MAP
        self.lifes -= 1
        # remove bonus
        MAP[17][13] = 0
        i = 1
        self.pacman.Q[self.pacman.current_state, self.pacman.action] = 0.99 * self.pacman.Q[self.pacman.current_state, self.pacman.action] - 0.01 * 1000
        while i < 15:
            # evaluate the pygame event
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    break
            self.clear_all_surfaces()
            self.display_map(self.surface)
            if i<11:
                self.surface.blit(self.Dead_pacman[i], (self.pacman.rect.x, self.pacman.rect.y))
            else:
                if i%2:
                    self.surface.blit(self.Dead_pacman[10], (self.pacman.rect.x, self.pacman.rect.y))
            self.display_lifes(self.bottom)
            self.blit_all_surfaces()
            pygame.display.flip()
            i += 1
            time.sleep(0.001)
        pygame.display.update()
        self.pacman.reinit(PACMAN_POS[0], PACMAN_POS[1])
        for ghost in self.Ghosts:
            ghost.reinit(JAIL[ghost.color][0], JAIL[ghost.color][1], "jail")
        if self.lifes > 0:
            self.start_game()
        self.current_mode_idx = 0
        self.start_mode_timer = time.time()

    def clear_all_surfaces(self):
        self.fake_screen.fill(BLACK)
        self.surface.fill(BLACK)
        self.top.fill(BLACK)
        self.bottom.fill(BLACK)

    def blit_all_surfaces(self):
        self.fake_screen.blit(self.top, (0, 0))
        self.fake_screen.blit(self.surface, (0, 32))
        self.fake_screen.blit(self.bottom, (0, self.HEIGHT+32))
        self.screen.blit(self.scale_output(self.fake_screen, self.scale), (0, 0))

    def scale_output(self, my_surface, my_scale):
        if my_scale != 1:
            frame = pygame.transform.scale(my_surface, (int(self.FULL_WIDTH * my_scale), int(self.FULL_HEIGHT * my_scale)))
        else:
            frame = my_surface
        return frame

    def count_pacgums(self):
        global MAP
        y_length = len(MAP)
        x_length = len(MAP[0])
        self.pacgums = 0
        for y in range(y_length):
            for x in range(x_length):
                if MAP[y][x] in (1, 2):
                    self.pacgums += 1
        return self.pacgums

    def display_map(self, my_surface):
        global MAP
        y_length = len(MAP)
        x_length = len(MAP[0])
        for y in range(y_length):
            for x in range(x_length):
                c = MAP[y][x]
                if c in self.Walls:
                    if c != 2:
                        my_surface.blit(self.Walls[c], (x*24, y*24))
                    else:
                        if self.count_loops%8 in range(0,3):
                            my_surface.blit(self.Walls[c], (x*24, y*24))

    def collided(self):
        hit_list = pygame.sprite.spritecollide(self.pacman, self.all_ghosts, False, pygame.sprite.collide_circle)
        return hit_list

    def play(self):
        self.ghosts_in_a_row = 0
        lives_gained = 0
        clock = pygame.time.Clock()
        running = True
        while running and self.level <= 1:
            self.count_loops = 0
            self.start_game()
            self.level += 1
            self.current_mode_idx = 0
            self.current_mode = LEVELS[self.level]['MODES'][self.current_mode_idx][0]
            self.start_mode_timer = time.time()
            while running:
                clock.tick(self.FPS)
                self.count_loops += 1
                
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        running = False
                current_time = time.time()
                if current_time - self.start_mode_timer > LEVELS[self.level]['MODES'][self.current_mode_idx][1]:
                    self.start_mode_timer = current_time
                    self.current_mode_idx += 1
                    self.current_mode = LEVELS[self.level]['MODES'][self.current_mode_idx][0]
                self.pacman.update()
                self.all_ghosts.update()
                if self.count_loops % 2 == 0:
                    # print(self.score)
                    self.display_board_game()
                pygame.display.flip()

                # Collision test
                hit_list = self.collided()
                if hit_list:
                    if self.pacman.mode == "chase":
                        for ghost in hit_list:
                            if ghost.mode == "runaway":
                                ghost.change_mode("eaten")
                                self.ghosts_in_a_row += 1
                                ghost_score = 200*math.pow(2, self.ghosts_in_a_row-1)
                                self.score += ghost_score
                            elif ghost.mode != "eaten":
                                self.loose_life()
                    else:
                        for ghost in hit_list:
                            if ghost.mode not in ("eaten", "runaway"):
                                self.loose_life()
                                break
                if self.pacgums == 0:
                    running = False
                    self.win = True
                if self.lifes == 0:
                    running = False
                if self.count_loops > self.max_count_loops:
                    running = False
                    self.trunc = True
        return (self.pacman.get_q_table(), self.score, self.count_loops, self.win, self.trunc)

# Root code
def main():
    global MAP
    
    # Инициализация окна OpenCV для визуализации
    cv2.namedWindow('Computer Vision Analysis', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Computer Vision Analysis', 600, 800)

    # Инициализация Q-таблицы с нуля
    # Количество состояний: (8 направлений к призраку) * (8 направлений к точке) * (3 варианта allowed_moves)
    num_states = 8 * 8 * 3  # 192 состояния
    input_q_table = np.zeros((num_states, 5))
    
    # Параметры обучения
    num_episodes = 1
    scores = []
    
    for episode in range(num_episodes):
        MAP = copy.deepcopy(MAP_global)
        game = Game(input_q_table, 10000)
        
        # Обновляем Q-таблицу после игры
        input_q_table, score_, iteration_, win_, trunc_ = game.play()
        scores.append(score_)
        
        # Выводим прогресс
        if episode % 10 == 0:
            avg_score = np.mean(scores[-10:]) if len(scores) >= 10 else np.mean(scores)
            print(f"Episode {episode}, Score: {score_}, Avg (last 10): {avg_score:.1f}")
    
    pygame.quit()
    # Сохраняем обученную таблицу
    with open("q_table_trained.pkl", "wb") as file:
        pickle.dump(input_q_table, file)
    
    # Выводим статистику
    print(f"\nTraining complete!")
    print(f"Best score: {max(scores)}")
    print(f"Average score: {np.mean(scores):.1f}")
        
if __name__ == "__main__":
    main()